**Scenario**

A retail company maintains a product catalog in its data warehouse. Product details such as name, category, and price may change over time due to rebranding, category updates, or pricing adjustments. To preserve historical data for accurate reporting and trend analysis, the company needs to implement a Slowly Changing Dimension (SCD) Type 2 mechanism in PySpark, ensuring old records are retained with effective date ranges while new versions are inserted as separate records.


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ==================================================
# 1. CREATE SOURCE TABLE
# ==================================================

spark.sql("""
CREATE TABLE IF NOT EXISTS pyspark_cata.source.customers
(
id STRING,
email STRING,
city STRING,
country STRING,
modifiedDate TIMESTAMP
)
""")

# ==================================================
# 2. INSERT INITIAL DATA
# ==================================================

spark.sql("""
INSERT INTO pyspark_cata.source.customers
VALUES
('1','john.smith@example.com','New York','USA',current_timestamp()),
('2','jane.doe@example.com','London','UK',current_timestamp()),
('3','mike.williams@example.com','Paris','France',current_timestamp()),
('4','sara.jones@example.com','Tokyo','Japan',current_timestamp()),
('5','peter.chen@example.com','Sydney','Australia',current_timestamp())
""")

display(spark.sql("""
SELECT *
FROM pyspark_cata.source.customers
"""))

# ==================================================
# 3. INITIAL LOAD TO DIMENSION
# ==================================================

if spark.catalog.tableExists("pyspark_cata.source.DimCustomers"):
pass
else:
spark.sql("""
CREATE TABLE pyspark_cata.source.DimCustomers
AS
SELECT *,
current_timestamp() AS startTime,
CAST('3000-01-01' AS TIMESTAMP) AS endTime,
'Y' AS isActive
FROM pyspark_cata.source.customers
""")

display(spark.sql("""
SELECT *
FROM pyspark_cata.source.DimCustomers
"""))

# ==================================================
# 4. INSERT UPDATED + NEW RECORDS
# ==================================================

spark.sql("""
INSERT INTO pyspark_cata.source.customers
VALUES
('1','john.smith@example.com','Seattle','USA',current_timestamp()),
('6','jane.doe@example.com','London','UK',current_timestamp())
""")

display(spark.sql("""
SELECT *
FROM pyspark_cata.source.customers
"""))

# ==================================================
# 5. DEDUPLICATE SOURCE
# ==================================================

df = spark.sql("""
SELECT *
FROM pyspark_cata.source.customers
""")

df = df.withColumn(
"dedup",
row_number().over(
Window.partitionBy("id")
.orderBy(desc("modifiedDate"))
)
)

df = df.filter(col("dedup") == 1)

df = df.drop("dedup")

df.createOrReplaceTempView("srctemp")

# ==================================================
# 6. PREPARE SOURCE VIEW
# ==================================================

df = spark.sql("""
SELECT *,
current_timestamp() AS startTime,
CAST('3000-01-01' AS TIMESTAMP) AS endTime,
'Y' AS isActive
FROM srctemp
""")

df.createOrReplaceTempView("src")

display(spark.sql("""
SELECT *
FROM src
"""))

# ==================================================
# 7. MERGE-1
# MARK UPDATED RECORDS AS EXPIRED
# ==================================================

spark.sql("""
MERGE INTO pyspark_cata.source.DimCustomers AS trg
USING src AS src

ON trg.id = src.id
AND trg.isActive = 'Y'

WHEN MATCHED AND
(
src.email <> trg.email
OR src.city <> trg.city
OR src.country <> trg.country
OR src.modifiedDate <> trg.modifiedDate
)

THEN UPDATE SET
trg.endTime = current_timestamp(),
trg.isActive = 'N'
""")

# CHECK AFTER MERGE-1

display(spark.sql("""
SELECT *
FROM pyspark_cata.source.DimCustomers
ORDER BY id,startTime
"""))

# ==================================================
# 8. MERGE-2
# INSERT NEW + UPDATED RECORDS
# ==================================================

spark.sql("""
MERGE INTO pyspark_cata.source.DimCustomers AS trg
USING src AS src

ON src.id = trg.id
AND trg.isActive = 'Y'

WHEN NOT MATCHED
THEN INSERT *
""")

# ==================================================
# 9. FINAL OUTPUT
# ==================================================

display(spark.sql("""
SELECT *
FROM pyspark_cata.source.DimCustomers
ORDER BY id,startTime
"""))
